# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata (name and description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Optionally: pretty print metadata
pp = pprint.PrettyPrinter(indent=2)
print("\nFull metadata overview:")
pp.pprint(dataset.metadata.to_json())

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

**Note:** In Croissant, `recordSet` and related IDs describe the structure and content units of the dataset. We'll inspect the record sets, their fields, and columns.

In [ ]:
# Explore record sets, fields, and columns by @id
record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No record sets available in this dataset - please check Croissant metadata.")
else:
    print("Record Sets:")
    for rs in record_sets:
        print(f"- Record Set @id: {rs['@id']} | name: {rs.get('name', '')}")
        print("  Fields:")
        for field in rs.get('fields', []):
            print(f"    - Field @id: {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType', '')}")
        print("  Columns:")
        for column in rs.get('columns', []):
            print(f"    - Column @id: {column['@id']} | name: {column.get('name', '')} | source: {column.get('source', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

For this example, we will extract the main survey records.

**Note:** Replace the IDs below appropriately if the structure changes, but these follow the `@id` referencing rule.

In [ ]:
# If record sets are empty (as in this schema), mlcroissant may infer record set from distribution.

# We'll inspect the distribution for the files
distributions = dataset.metadata.distribution
print("Distributions:")
for dist in distributions:
    print(f"- Distribution @id: {dist['@id']}")

# If record sets are defined, use their @id. Otherwise, attempt loading records from available distributions
# (assuming mlcroissant will expose inferred record_set IDs, or allow direct loading)
record_set_ids = []
if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    # For example: recordSet might be inferred from file/distribution @id
    record_set_ids = [dist['@id'] for dist in distributions]

# Load records from each record set
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for Record Set @id: {record_set_id}")
        print(e)

# For later steps, let's pick one record set @id (the first, if available)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We'll reference columns and fields by their `@id` (using DataFrame column names corresponding to field/column names).

**Example:** Suppose the survey includes numeric scores, e.g. GAD-7, PHQ-9, PSQ, and demographics for grouping.

In [ ]:
# Choose a numeric field (update as necessary based on dataset structure)
# Common survey fields: 'gad_7_score', 'phq_9_score', 'psq_score', 'age', etc.

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print("Available columns:", df.columns.tolist())

    # Example using 'gad_7_score'. Replace with actual @id field name if column differs.
    numeric_field = 'gad_7_score' if 'gad_7_score' in df.columns else None

    if numeric_field:
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize values
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by 'gender' field (ensure it exists and replace by actual @id if needed)
        group_field = 'gender' if 'gender' in df.columns else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No group field found for grouping (e.g., 'gender').")
    else:
        print("No numeric field found for EDA (e.g., 'gad_7_score').")
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of a numeric mental health score, and optionally group by a demographic feature.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    numeric_field = 'gad_7_score' if 'gad_7_score' in df.columns else None
    group_field = 'gender' if 'gender' in df.columns else None

    if numeric_field:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field], bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

        if group_field:
            plt.figure(figsize=(8,5))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
    else:
        print("No numeric field found for visualization.")
else:
    print("No main record set DataFrame available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides comprehensive coverage of mental health indicators in Kilifi County, Kenya.
- Basic exploratory analysis reflects distributions of scores and highlights demographic features that may impact mental health.
- The Croissant metadata structure enables AI-ready, standardized access, allowing robust processing and reproducibility.
- For deeper research, use additional fields and advanced grouping or modeling available via the dataset schema.

_End of notebook._